# 04 · Silver — Cotizaciones

Unifica las cotizaciones de todas las fuentes en `silver.cotizaciones`, con precios comparables a los operados.

```
bronze.data912_cotizaciones ─ conform_data912: renta fija ÷ 100 (cotiza cada 100 VN) ─┐
                                                                                        ├─▶ silver.cotizaciones (simbolo, fecha, fuente)
bronze.yahoo_cotizaciones   ─ conform_yahoo: revierte splits posteriores ──────────────┘   es_preferida = menor prioridad del registro
```

| Regla | Criterio | Decisión |
|---|---|---|
| Última extracción | Por (símbolo, fecha) se usa la extracción más reciente; si un símbolo falló en la última, se conserva el dato anterior | DT-021 |
| Escala | data912 publica la renta fija cada 100 VN; el dataset, por 1 VN | DT-021 |
| Splits | Yahoo suele ajustar retroactivamente: precio real = publicado × splits posteriores. Solo se revierten los splits que Yahoo aplicó a la historia | DT-021 |
| Fuente preferida | Por (símbolo, fecha), la de menor prioridad para el tipo de instrumento; las demás se conservan | DT-019 |

**Prerrequisito:** `setup/06_cotizaciones.sql` y el notebook `03_cotizaciones_ingesta`.

## Parámetros

In [ ]:
dbutils.widgets.text("catalog", "workspace", "Catálogo")
CATALOG = dbutils.widgets.get("catalog")

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone

# La lógica vive en src/ para poder testearla sin cluster (ver tests/).
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from pyspark.sql import functions as F

from iol_pipeline.config import PipelineConfig
from iol_pipeline.silver_cotizaciones import MIN_DETECTABLE_SPLIT, build_silver_quotes, yahoo_split_events
from iol_pipeline.sources import QUOTE_SOURCES

spark.conf.set("spark.sql.session.timeZone", "UTC")

cfg = PipelineConfig(catalog=CATALOG)
silver_table = cfg.silver_table("transacciones")
quotes_table = cfg.silver_table("cotizaciones")
bronze_tables = {s.name: cfg.bronze_table(s.bronze_table) for s in QUOTE_SOURCES}

for name, table in bronze_tables.items():
    print(f"{name:8} {table}")
print(f"destino  {quotes_table}")

## Precondiciones

In [ ]:
for table in [silver_table, quotes_table, *bronze_tables.values()]:
    assert spark.catalog.tableExists(table), f"No existe {table}. Ejecutar setup/06_cotizaciones.sql"

bronze_rows = {name: spark.table(t).count() for name, t in bronze_tables.items()}
print("Filas en bronze por fuente:", bronze_rows)
assert sum(bronze_rows.values()) > 0, "No hay cotizaciones en bronze. Ejecutar 03_cotizaciones_ingesta"

## Normalización y fuente preferida

In [ ]:
silver = spark.table(silver_table)
instrument_types = silver.select("simbolo", "tipo_instrumento").distinct()
processed_at = datetime.now(timezone.utc).replace(tzinfo=None)

build_silver_quotes(spark, bronze_tables, instrument_types, QUOTE_SOURCES, processed_at).writeTo(quotes_table).overwrite(F.lit(True))
quotes = spark.table(quotes_table)

display(
    quotes.groupBy("fuente")
    .agg(F.count("*").alias("filas"), F.sum(F.col("es_preferida").cast("int")).alias("preferidas"),
         F.countDistinct("simbolo").alias("simbolos"), F.min("fecha").alias("desde"), F.max("fecha").alias("hasta"))
    .orderBy("fuente")
)

## Reversión de splits (Yahoo)
Yahoo informa los splits, pero **no siempre ajusta los precios previos** (ej. MIRG). Por eso se detecta con los datos: si la serie es continua en la fecha del split (`salto_de_precio` ≈ 1), Yahoo ajustó la historia y el split se revierte; si el precio salta en el tamaño del factor, no la ajustó y no se toca. Con factores chicos (dividendos en acciones) o sin cierre ese día, se asume ajustada.

In [ ]:
print(f"Factor mínimo detectable: {MIN_DETECTABLE_SPLIT}x")
display(yahoo_split_events(spark.table(bronze_tables["yahoo"])).dropDuplicates(["simbolo", "fecha_split"]).orderBy("simbolo", "fecha_split"))

## Consistencia entre fuentes
Donde ambas fuentes cotizan el mismo símbolo y día, sus cierres normalizados deberían coincidir. Es la validación independiente de la reversión de splits y de la escala.

In [ ]:
pairs = (
    quotes.where("fuente = 'data912'").select("simbolo", "fecha", F.col("cierre").alias("cierre_data912"))
    .join(quotes.where("fuente = 'yahoo'").select("simbolo", "fecha", F.col("cierre").alias("cierre_yahoo"), "factor_ajuste"),
          ["simbolo", "fecha"])
    .withColumn("dif_pct", F.abs(F.col("cierre_yahoo") / F.col("cierre_data912") - 1) * 100)
)
display(pairs.agg(
    F.countDistinct("simbolo").alias("simbolos_en_ambas"),
    F.count("*").alias("dias_comparados"),
    F.round(F.expr("percentile(dif_pct, 0.5)"), 3).alias("dif_mediana_pct"),
    F.round(F.expr("percentile(dif_pct, 0.9)"), 3).alias("dif_p90_pct"),
    F.sum((F.col("dif_pct") > 10).cast("int")).alias("dias_con_dif_mayor_10pct"),
))

Ejemplo: símbolos con split, el mismo día en ambas fuentes.

In [ ]:
display(
    pairs.where("factor_ajuste <> 1").orderBy("simbolo", "fecha")
    .groupBy("simbolo").agg(F.first("fecha").alias("fecha"), F.first("factor_ajuste").alias("factor_split"),
                            F.first("cierre_yahoo").alias("cierre_yahoo_revertido"), F.first("cierre_data912").alias("cierre_data912"))
)

## Escala contra lo operado
Mediana de `precio operado / cierre preferido` por fuente y tipo de instrumento, en el mismo símbolo y fecha operativa. Debe dar cerca de 1: confirma que precio operado y cierre están en la misma unidad.

In [ ]:
preferred = quotes.where("es_preferida").select("simbolo", F.col("fecha").alias("fecha_operativa"), "fuente", "cierre")
display(
    silver.where("dq_status = 'OK'").join(preferred, ["simbolo", "fecha_operativa"])
    .withColumn("ratio", F.col("precio").cast("double") / F.col("cierre"))
    .groupBy("fuente", "tipo_instrumento")
    .agg(F.count("*").alias("operaciones"), F.round(F.expr("percentile(ratio, 0.5)"), 4).alias("ratio_mediana"),
         F.round(F.expr("percentile(ratio, 0.1)"), 4).alias("ratio_p10"), F.round(F.expr("percentile(ratio, 0.9)"), 4).alias("ratio_p90"))
    .orderBy("fuente", F.desc("operaciones"))
)

## Resultado

In [ ]:
rows = quotes.count()
summary = {
    "table": quotes_table,
    "rows": rows,
    "status": "OK" if rows > 0 else "FAIL",
    "message": f"{rows} cotizaciones normalizadas de {len(QUOTE_SOURCES)} fuentes" if rows else "No se generaron cotizaciones",
}
print(json.dumps(summary, indent=2))
dbutils.notebook.exit(json.dumps(summary))